### Realizamos la conexion hacia nuestro ADLS por medio de service principal

In [0]:
%run ../config/Access_ADLS_Service_Principal

In [0]:
%run ../includes/configuration

In [0]:
%run ../includes/common_functions

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

### Leemos el archivo movie.csv de nuestro contenedor bronze

In [0]:
# Se accede a la ruta del contenedor desde la variable del notebook/includes/configuration
df = spark.read.option("header", "true")\
          .option("inferSchema", "true")\
          .csv(f"{bronze_folder_path}/movie.csv")

###### Borramos las columnas que no nos interesan

In [0]:
from pyspark.sql.functions import col,current_timestamp, lit
df_drop = df.drop(col("homePage"))\
            .drop(col("overview"))\
            .drop(col("movieStatus"))\
            .drop(col("tagline"))

###### Cambiamos el nombre de las columnas

In [0]:
df_renamed = df_drop.withColumnRenamed("movieId", "movie_id")\
                    .withColumnRenamed("yearReleaseDate", "year_release_date")\
                    .withColumnRenamed("releaseDate", "release_date")\
                    .withColumnRenamed("durationTime", "duration_time")\
                    .withColumnRenamed("voteAverage", "vote_average")\
                    .withColumnRenamed("voteCount", "vote_count")

###### Adiccionamos dos nuevas columnas, una para guardar la fecha de ingestion y la otra para guardar el ambiente

In [0]:
#Addicionamos las dos columnas de auditoria por medio de una funcion de otro notebook
df_add = add_columnas_control(df_renamed, v_environment)

###### Escribimos los datos en nuestro contenedor silver del data lake

In [0]:
df_add.write.mode("overwrite").parquet(f"{silver_folder_path}/movie")